---
title: "Lab 1: Table Partitioning in PostgreSQL (Short Version)"
author: "Dr. Tim Smith"
date: "March 4, 2026"
format:
  html:
    theme: cosmo
    toc: true
    toc-depth: 3
    toc-location: left
    code-fold: false
    code-copy: true
    embed-resources: true
    number-sections: true
    page-layout: full
execute:
  eval: false
---

## Overview

This is the condensed in-class version of the partitioning lab. It covers range partitioning, list partitioning, partition pruning, and instant data removal. For hash partitioning, PK constraints, DEFAULT partitions, per-partition indexes, and maintenance operations, see the **comprehensive lab**.

**Prerequisites:** Start the Docker environment with `cd partitioning-lab && docker compose up -d` and wait ~30 seconds for the 500K rows to load.

## Database Schema

The Docker init script creates three tables with the following structure. The `sensor_readings` table (500K rows) is the focus of this lab --- it has a simple `SERIAL PRIMARY KEY` on `reading_id` and **no additional indexes**.

<div style="display: flex; justify-content: center; gap: 2em; flex-wrap: wrap; margin: 1.5em 0;">
<div style="border: 2px solid #6c3483; border-radius: 6px; min-width: 240px; font-size: 0.85em; background: #f9f5fc;">
<div style="background: #6c3483; color: white; padding: 6px 12px; font-weight: bold; text-align: center;">locations <span style="font-weight: normal; font-size: 0.85em;">(10 rows)</span></div>
<table style="width: 100%; margin: 0; border: none;">
<tr><td style="border: none; padding: 3px 12px;">🔑 <code>location_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">SERIAL PK</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>building</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(50)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>floor</code></td><td style="border: none; padding: 3px 12px; color: #888;">INTEGER</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>room</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(20)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>description</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(200)</td></tr>
</table>
</div>
<div style="border: 2px solid #2874a6; border-radius: 6px; min-width: 260px; font-size: 0.85em; background: #f0f7fc;">
<div style="background: #2874a6; color: white; padding: 6px 12px; font-weight: bold; text-align: center;">sensors <span style="font-weight: normal; font-size: 0.85em;">(50 rows)</span></div>
<table style="width: 100%; margin: 0; border: none;">
<tr><td style="border: none; padding: 3px 12px;">🔑 <code>sensor_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">SERIAL PK</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>sensor_type</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(20)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>model</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(50)</td></tr>
<tr><td style="border: none; padding: 3px 12px;">🔗 <code>location_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">FK → locations</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>installed_date</code></td><td style="border: none; padding: 3px 12px; color: #888;">DATE</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>is_active</code></td><td style="border: none; padding: 3px 12px; color: #888;">BOOLEAN</td></tr>
</table>
</div>
<div style="border: 2px solid #c0392b; border-radius: 6px; min-width: 280px; font-size: 0.85em; background: #fdf2f0;">
<div style="background: #c0392b; color: white; padding: 6px 12px; font-weight: bold; text-align: center;">sensor_readings <span style="font-weight: normal; font-size: 0.85em;">(500,000 rows)</span></div>
<table style="width: 100%; margin: 0; border: none;">
<tr><td style="border: none; padding: 3px 12px;">🔑 <code>reading_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">SERIAL PK</td></tr>
<tr><td style="border: none; padding: 3px 12px;">🔗 <code>sensor_id</code></td><td style="border: none; padding: 3px 12px; color: #888;">FK → sensors</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>reading_time</code></td><td style="border: none; padding: 3px 12px; color: #888;">TIMESTAMP</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>value</code></td><td style="border: none; padding: 3px 12px; color: #888;">NUMERIC(10,2)</td></tr>
<tr><td style="border: none; padding: 3px 12px;"><code>unit</code></td><td style="border: none; padding: 3px 12px; color: #888;">VARCHAR(10)</td></tr>
</table>
</div>
</div>

::: {.callout-note title="Why This Matters for Partitioning"}
The `sensor_readings` table has a single-column `PRIMARY KEY (reading_id)`. When we partition this table later, we'll need to change the PK to include the partition key (e.g., `PRIMARY KEY (reading_id, reading_time)`), because PostgreSQL cannot enforce uniqueness across partitions.
:::

## Setup and Connection

In [ ]:
import psycopg2
import time

# Connection parameters
CONN_PARAMS = {
    'host': 'localhost',
    'port': 5502,
    'dbname': 'sensor_db',
    'user': 'student',
    'password': 'student'
}

conn = psycopg2.connect(**CONN_PARAMS)
conn.autocommit = True
cur = conn.cursor()

print("Connected to sensor_db on port 5502")

```text
Connected to sensor_db on port 5502
```

In [ ]:
# Verify the data loaded correctly
cur.execute("SELECT COUNT(*) FROM sensor_readings")
total = cur.fetchone()[0]

cur.execute("SELECT MIN(reading_time), MAX(reading_time) FROM sensor_readings")
min_ts, max_ts = cur.fetchone()

cur.execute("SELECT COUNT(DISTINCT sensor_id) FROM sensor_readings")
sensor_count = cur.fetchone()[0]

print(f"Total readings:  {total:,}")
print(f"Sensors:         {sensor_count}")
print(f"Date range:      {min_ts.strftime('%Y-%m-%d')} to {max_ts.strftime('%Y-%m-%d')}")
print(f"\nThat's {total // sensor_count:,} readings per sensor — about {total // sensor_count // 24} days of hourly data.")

```text
Total readings:  500,000
Sensors:         50
Date range:      2025-01-01 to 2026-02-21

That's 10,000 readings per sensor — about 416 days of hourly data.
```

## Baseline: Querying the Non-Partitioned Table

Before partitioning, let's see how PostgreSQL handles a month-filter query on the full 500K-row table using `EXPLAIN ANALYZE`.

In [ ]:
def explain_analyze(cursor, query, params=None):
    """Run EXPLAIN ANALYZE and print the query plan."""
    cursor.execute(f"EXPLAIN ANALYZE {query}", params)
    plan = cursor.fetchall()
    for row in plan:
        print(row[0])
    return plan

In [ ]:
# Baseline: Query readings for a single month
print("BASELINE: All readings for March 2025 (non-partitioned)")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM sensor_readings
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-04-01'
""")

print("\n** Notice: PostgreSQL must scan the ENTIRE table to find March data.")
print("   Each of the 3 parallel workers scanned ~166K rows (12,400 kept +")
print("   154,267 discarded = ~166,667 per worker × 3 = ~500K total).")
print("   There's no way to skip irrelevant months — it's all one big table.")

```text
BASELINE: All readings for March 2025 (non-partitioned)
============================================================
Finalize Aggregate  (cost=7880.27..7880.28 rows=1 width=40) (actual time=9.005..11.286 rows=1 loops=1)
  ->  Gather  (cost=7880.04..7880.25 rows=2 width=40) (actual time=8.994..11.276 rows=3 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        ->  Partial Aggregate  (cost=6880.04..6880.05 rows=1 width=40) (actual time=7.291..7.291 rows=1 loops=3)
              ->  Parallel Seq Scan on sensor_readings  (cost=0.00..6802.00 rows=15608 width=6) (actual time=0.715..6.649 rows=12400 loops=3)
                    Filter: ((reading_time >= '2025-03-01 00:00:00'::timestamp without time zone) AND (reading_time < '2025-04-01 00:00:00'::timestamp without time zone))
                    Rows Removed by Filter: 154267
Planning Time: 0.115 ms
Execution Time: 11.310 ms

** Notice: PostgreSQL must scan the ENTIRE table to find March data.
   Each of the 3 parallel workers scanned ~166K rows (12,400 kept +
   154,267 discarded = ~166,667 per worker × 3 = ~500K total).
   There's no way to skip irrelevant months — it's all one big table.
```

::: {.callout-important title="Key Observation"}
PostgreSQL reads the entire table across 3 parallel workers. Each worker scanned ~166,667 rows (12,400 returned + 154,267 discarded), totaling all 500K rows. Execution time: **~11.3 ms**.
:::

## Range Partitioning (by Month)

Range partitioning splits a table by value ranges --- ideal for time-series data. In practice, you don't rebuild a table from scratch --- you **retrofit partitioning onto the existing table** when it gets too large. Here's how that migration works in PostgreSQL.

<div style="text-align: center; margin: 1.5em 0;">
<svg viewBox="0 0 800 200" style="max-width: 800px; font-family: sans-serif;">
  <rect x="10" y="60" width="200" height="80" rx="8" fill="#e8daef" stroke="#6c3483" stroke-width="2"/>
  <text x="110" y="95" text-anchor="middle" font-weight="bold" font-size="14">sensor_readings</text>
  <text x="110" y="115" text-anchor="middle" font-size="12" fill="#555">500K rows</text>
  <text x="320" y="55" text-anchor="middle" font-size="11" fill="#6c3483" font-weight="bold">PARTITION BY RANGE</text>
  <text x="320" y="70" text-anchor="middle" font-size="11" fill="#6c3483">(reading_time)</text>
  <line x1="210" y1="100" x2="380" y2="45" stroke="#6c3483" stroke-width="1.5" marker-end="url(#arrow-part-s)"/>
  <line x1="210" y1="100" x2="380" y2="95" stroke="#6c3483" stroke-width="1.5" marker-end="url(#arrow-part-s)"/>
  <line x1="210" y1="100" x2="380" y2="145" stroke="#6c3483" stroke-width="1.5" marker-end="url(#arrow-part-s)"/>
  <line x1="210" y1="100" x2="380" y2="185" stroke="#6c3483" stroke-width="1.5" marker-end="url(#arrow-part-s)"/>
  <defs><marker id="arrow-part-s" viewBox="0 0 10 10" refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" fill="#6c3483"/></marker></defs>
  <rect x="390" y="20" width="180" height="45" rx="6" fill="#d5f5e3" stroke="#27ae60" stroke-width="1.5"/>
  <text x="480" y="40" text-anchor="middle" font-weight="bold" font-size="11">readings_2025_01</text>
  <text x="480" y="55" text-anchor="middle" font-size="10" fill="#555">37,200 rows</text>
  <rect x="390" y="75" width="180" height="45" rx="6" fill="#d5f5e3" stroke="#27ae60" stroke-width="1.5"/>
  <text x="480" y="95" text-anchor="middle" font-weight="bold" font-size="11">readings_2025_02</text>
  <text x="480" y="110" text-anchor="middle" font-size="10" fill="#555">33,600 rows</text>
  <rect x="390" y="125" width="180" height="45" rx="6" fill="#d5f5e3" stroke="#27ae60" stroke-width="1.5"/>
  <text x="480" y="145" text-anchor="middle" font-weight="bold" font-size="11">readings_2025_03</text>
  <text x="480" y="160" text-anchor="middle" font-size="10" fill="#555">37,200 rows</text>
  <text x="480" y="192" text-anchor="middle" font-size="12" fill="#888">... (12 more)</text>
</svg>
<p style="font-size: 0.85em; color: #555; margin-top: 0.5em;">Range partitioning: queries that filter on <code>reading_time</code> only scan the relevant partition(s).</p>
</div>

### Step 1: Rename the Existing Table

The first step in a real migration is to **move the existing table out of the way**. We rename it so we can create a new partitioned table with the original name. Any application code using `sensor_readings` will temporarily break until we complete the swap.

In [ ]:
# Step 1: Rename the existing non-partitioned table
cur.execute("ALTER TABLE sensor_readings RENAME TO sensor_readings_old")

# Verify the rename
cur.execute("SELECT COUNT(*) FROM sensor_readings_old")
count = cur.fetchone()[0]
print(f"Renamed sensor_readings -> sensor_readings_old ({count:,} rows)")
print("\nThe original data is safe — we haven't deleted anything.")

```text
Renamed sensor_readings -> sensor_readings_old (500,000 rows)

The original data is safe — we haven't deleted anything.
```

### Step 2: Create the Partitioned Replacement

Now we create a new `sensor_readings` table **with the same columns** but with `PARTITION BY RANGE`. Note the PK changes from `(reading_id)` to `(reading_id, reading_time)` --- PostgreSQL requires the partition key in any primary key constraint.

In [ ]:
# Step 2: Create partitioned table with the SAME name
cur.execute("""
    CREATE TABLE sensor_readings (
        reading_id    INTEGER        NOT NULL,
        sensor_id     INTEGER        NOT NULL,
        reading_time  TIMESTAMP      NOT NULL,
        value         NUMERIC(10,2)  NOT NULL,
        unit          VARCHAR(10)    NOT NULL,
        PRIMARY KEY (reading_id, reading_time)
    ) PARTITION BY RANGE (reading_time)
""")

# Create monthly partitions from Jan 2025 through Mar 2026
months = []
year, month = 2025, 1
while (year, month) <= (2026, 3):
    next_month = month + 1
    next_year = year
    if next_month > 12:
        next_month = 1
        next_year = year + 1

    partition_name = f"readings_{year}_{month:02d}"
    start_date = f"{year}-{month:02d}-01"
    end_date = f"{next_year}-{next_month:02d}-01"

    cur.execute(f"""
        CREATE TABLE {partition_name} PARTITION OF sensor_readings
        FOR VALUES FROM ('{start_date}') TO ('{end_date}')
    """)
    months.append(partition_name)

    month = next_month
    year = next_year

print(f"Created partitioned sensor_readings with {len(months)} monthly partitions:")
for m in months:
    print(f"  {m}")

```text
Created partitioned sensor_readings with 15 monthly partitions:
  readings_2025_01
  readings_2025_02
  readings_2025_03
  readings_2025_04
  readings_2025_05
  readings_2025_06
  readings_2025_07
  readings_2025_08
  readings_2025_09
  readings_2025_10
  readings_2025_11
  readings_2025_12
  readings_2026_01
  readings_2026_02
  readings_2026_03
```

### Step 3: Migrate Data from the Old Table

Now we move the 500K rows from the old non-partitioned table into the new partitioned one. PostgreSQL automatically routes each row to the correct monthly partition based on `reading_time`.

In [ ]:
# Step 3: Migrate data from old table into partitioned table
start = time.time()

cur.execute("""
    INSERT INTO sensor_readings (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings_old
""")

elapsed = time.time() - start

cur.execute("SELECT COUNT(*) FROM sensor_readings")
count = cur.fetchone()[0]
print(f"Migrated {count:,} rows into partitioned table in {elapsed:.1f}s\n")

# Check distribution
print("Rows per partition:")
print("=" * 40)
for partition in months:
    cur.execute(f"SELECT COUNT(*) FROM {partition}")
    count = cur.fetchone()[0]
    if count > 0:
        bar = '#' * (count // 1000)
        print(f"  {partition:20s}  {count:>7,}  {bar}")

```text
Migrated 500,000 rows into partitioned table in 0.2s

Rows per partition:
========================================
  readings_2025_01       37,200  #####################################
  readings_2025_02       33,600  #################################
  readings_2025_03       37,200  #####################################
  readings_2025_04       36,000  ####################################
  readings_2025_05       37,200  #####################################
  readings_2025_06       36,000  ####################################
  readings_2025_07       37,200  #####################################
  readings_2025_08       37,200  #####################################
  readings_2025_09       36,000  ####################################
  readings_2025_10       37,200  #####################################
  readings_2025_11       36,000  ####################################
  readings_2025_12       37,200  #####################################
  readings_2026_01       37,200  #####################################
  readings_2026_02       24,800  ########################
```

### Step 4: Verify and Drop the Old Table

Once we've confirmed the data migrated correctly, we can safely drop the old table.

In [ ]:
# Step 4: Verify counts match, then drop the old table
cur.execute("SELECT COUNT(*) FROM sensor_readings")
new_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM sensor_readings_old")
old_count = cur.fetchone()[0]

print(f"Old table:         {old_count:,} rows")
print(f"Partitioned table: {new_count:,} rows")

if new_count == old_count:
    cur.execute("DROP TABLE sensor_readings_old")
    print(f"\nCounts match — dropped sensor_readings_old.")
    print("The partitioned table is now the live table with the original name.")
else:
    print(f"\nWARNING: counts don't match! Keeping old table.")

```text
Old table:         500,000 rows
Partitioned table: 500,000 rows

Counts match — dropped sensor_readings_old.
The partitioned table is now the live table with the original name.
```

::: {.callout-tip title="Real-World Migration Pattern"}
This rename → create → migrate → drop pattern is how partitioning is added in production. Application code still queries `sensor_readings` --- only the underlying storage changed. In a production setting, you'd wrap steps 1--3 in a maintenance window or use logical replication to minimize downtime.
:::

### Partition Pruning in Action

Now the same March 2025 query against the partitioned table. Watch for partition pruning --- PostgreSQL only scans `readings_2025_03`.

In [ ]:
print("PARTITIONED: All readings for March 2025")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM sensor_readings
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-04-01'
""")

print("\n** How to read this plan:")
print("   1. The scan line says 'Seq Scan on readings_2025_03' — that's the ONLY")
print("      partition mentioned. The other 14 partitions don't appear at all,")
print("      meaning PostgreSQL pruned (skipped) them before execution.")
print("   2. rows=37,200 with loops=1 — it scanned 37,200 rows total, compared")
print("      to ~500K in the non-partitioned baseline.")
print("   3. No 'Rows Removed by Filter' line — every row in this partition")
print("      matched the WHERE clause, so nothing was discarded.")
print("   4. No Parallel Seq Scan — the partition is small enough that PostgreSQL")
print("      didn't need parallel workers.")

```text
PARTITIONED: All readings for March 2025
============================================================
Aggregate  (cost=615.70..615.71 rows=1 width=40) (actual time=4.083..4.083 rows=1 loops=1)
  ->  Seq Scan on readings_2025_03 sensor_readings  (cost=0.00..615.13 rows=114 width=16) (actual time=0.004..2.420 rows=37200 loops=1)
        Filter: ((reading_time >= '2025-03-01 00:00:00'::timestamp without time zone) AND (reading_time < '2025-04-01 00:00:00'::timestamp without time zone))
Planning Time: 0.183 ms
Execution Time: 4.091 ms

** How to read this plan:
   1. The scan line says 'Seq Scan on readings_2025_03' — that's the ONLY
      partition mentioned. The other 14 partitions don't appear at all,
      meaning PostgreSQL pruned (skipped) them before execution.
   2. rows=37,200 with loops=1 — it scanned 37,200 rows total, compared
      to ~500K in the non-partitioned baseline.
   3. No 'Rows Removed by Filter' line — every row in this partition
      matched the WHERE clause, so nothing was discarded.
   4. No Parallel Seq Scan — the partition is small enough that PostgreSQL
      didn't need parallel workers.
```

::: {.callout-important title="3x Speedup"}
| Metric | Non-Partitioned | Partitioned |
|--------|---:|---:|
| Rows scanned | 500,000 | 37,200 |
| Execution Time | 11.3 ms | 4.1 ms |
| **Speedup** | | **~3x faster** |
:::

### What Happens WITHOUT the Partition Key?

Pruning only works when the `WHERE` clause includes the partition key. Filtering on `sensor_id` forces PostgreSQL to scan **every partition**.

In [ ]:
# NO PRUNING: Query by sensor_id (not the partition key)
print("NO PRUNING: Query by sensor_id on range-partitioned table")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM sensor_readings
    WHERE sensor_id = 10
""")

print("\n** How to read this plan:")
print("   1. 'Parallel Append' lists every partition (readings_2025_01,")
print("      readings_2025_03, ... all 14). None were pruned because")
print("      sensor_id is NOT the partition key.")
print("   2. rows=3333 with loops=3 — each of 3 workers found ~3,333")
print("      matching rows, totaling ~10,000 for sensor_id=10.")
print("   3. Execution time: 10.4 ms — nearly identical to the non-partitioned")
print("      baseline (11.3 ms). Partitioning gave NO benefit here.")
print("\n   Design rule: choose your partition key based on your most")
print("   common query filter. If queries don't include it, no pruning.")

```text
NO PRUNING: Query by sensor_id on range-partitioned table
============================================================
Finalize Aggregate  (cost=6953.41..6953.42 rows=1 width=40) (actual time=8.479..10.339 rows=1 loops=1)
  ->  Gather  (cost=6953.18..6953.39 rows=2 width=40) (actual time=8.410..10.331 rows=3 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        ->  Partial Aggregate  (cost=5953.18..5953.19 rows=1 width=40) (actual time=6.565..6.569 rows=1 loops=3)
              ->  Parallel Append  (cost=0.00..5950.00 rows=636 width=16) (actual time=0.006..6.361 rows=3333 loops=3)
                    ->  Parallel Seq Scan on readings_2025_01 ...
                    ->  Parallel Seq Scan on readings_2025_03 ...
                    ... (all 14 partitions scanned)
Planning Time: 0.244 ms
Execution Time: 10.375 ms

** How to read this plan:
   1. 'Parallel Append' lists every partition (readings_2025_01,
      readings_2025_03, ... all 14). None were pruned because
      sensor_id is NOT the partition key.
   2. rows=3333 with loops=3 — each of 3 workers found ~3,333
      matching rows, totaling ~10,000 for sensor_id=10.
   3. Execution time: 10.4 ms — nearly identical to the non-partitioned
      baseline (11.3 ms). Partitioning gave NO benefit here.

   Design rule: choose your partition key based on your most
   common query filter. If queries don't include it, no pruning.
```

::: {.callout-warning title="Design Rule"}
Choose your partition key based on your **most common query filter**. If queries don't filter on the partition key, no pruning occurs.
:::

## List Partitioning (by Sensor Unit)

List partitioning assigns specific values to specific partitions --- useful for **categorical data**. We'll partition by sensor unit type (`F`, `%`, `hPa`).

In [ ]:
# Create list-partitioned table, load data, and check distribution
cur.execute("DROP TABLE IF EXISTS readings_by_unit CASCADE")

cur.execute("""
    CREATE TABLE readings_by_unit (
        reading_id    INTEGER        NOT NULL,
        sensor_id     INTEGER        NOT NULL,
        reading_time  TIMESTAMP      NOT NULL,
        value         NUMERIC(10,2)  NOT NULL,
        unit          VARCHAR(10)    NOT NULL
    ) PARTITION BY LIST (unit)
""")

cur.execute("CREATE TABLE readings_temperature PARTITION OF readings_by_unit FOR VALUES IN ('F')")
cur.execute("CREATE TABLE readings_humidity PARTITION OF readings_by_unit FOR VALUES IN ('%')")
cur.execute("CREATE TABLE readings_pressure PARTITION OF readings_by_unit FOR VALUES IN ('hPa')")

# Load data
cur.execute("""
    INSERT INTO readings_by_unit (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
""")

# Check distribution
print("List-partitioned table — rows per partition:")
print("=" * 50)
for name, unit_val in [('readings_temperature', 'F'), ('readings_humidity', '%'), ('readings_pressure', 'hPa')]:
    cur.execute(f"SELECT COUNT(*) FROM {name}")
    count = cur.fetchone()[0]
    pct = count / 500000 * 100
    print(f"  {name:25s}  {count:>7,}  ({pct:.0f}%)")

print("\nDistribution matches our sensor mix: 20 temp + 20 humidity + 10 pressure")

```text
List-partitioned table — rows per partition:
==================================================
  readings_temperature       200,000  (40%)
  readings_humidity          200,000  (40%)
  readings_pressure          100,000  (20%)

Distribution matches our sensor mix: 20 temp + 20 humidity + 10 pressure
```

In [ ]:
# Demonstrate list partition pruning
print("LIST PARTITION PRUNING: Query only pressure readings")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_unit
    WHERE unit = 'hPa'
""")

print("\n** How to read this plan:")
print("   1. 'Seq Scan on readings_pressure' — only this partition appears.")
print("      Temperature and humidity partitions are completely absent (pruned).")
print("   2. rows=100,000 with loops=1 — scanned 100K pressure rows, not")
print("      the full 500K table.")
print("   3. No 'Rows Removed by Filter' — every row in this partition has")
print("      unit='hPa', so nothing was discarded.")

```text
LIST PARTITION PRUNING: Query only pressure readings
============================================================
Aggregate  (cost=1501.13..1501.14 rows=1 width=40) (actual time=11.990..11.991 rows=1 loops=1)
  ->  Seq Scan on readings_pressure readings_by_unit  (cost=0.00..1499.60 rows=305 width=16) (actual time=0.005..7.195 rows=100000 loops=1)
        Filter: ((unit)::text = 'hPa'::text)
Planning Time: 0.073 ms
Execution Time: 12.004 ms

** How to read this plan:
   1. 'Seq Scan on readings_pressure' — only this partition appears.
      Temperature and humidity partitions are completely absent (pruned).
   2. rows=100,000 with loops=1 — scanned 100K pressure rows, not
      the full 500K table.
   3. No 'Rows Removed by Filter' — every row in this partition has
      unit='hPa', so nothing was discarded.
```

## Instant Data Removal: DROP vs DELETE

One of partitioning's biggest operational wins: **dropping a partition is nearly instant** (metadata-only), while `DELETE` must scan and remove rows one by one.

In [ ]:
# Compare DELETE (row-by-row) vs DROP PARTITION (metadata-only)
# We'll use the partitioned sensor_readings table for both tests.

cur.execute("""
    SELECT COUNT(*) FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
jan_count = cur.fetchone()[0]
print(f"January 2025 readings: {jan_count:,}\n")

# Save Jan data so we can restore it after each test
cur.execute("""
    CREATE TEMP TABLE jan_backup AS
    SELECT * FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")

# Test 1: Time DELETE (row-by-row removal)
start = time.time()
cur.execute("""
    DELETE FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
delete_time = time.time() - start
print(f"DELETE {jan_count:,} rows: {delete_time:.3f}s")

# Re-insert from backup
cur.execute("""
    INSERT INTO sensor_readings (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM jan_backup
""")

# Test 2: Time DROP PARTITION (metadata-only)
start = time.time()
cur.execute("ALTER TABLE sensor_readings DETACH PARTITION readings_2025_01")
cur.execute("DROP TABLE readings_2025_01")
drop_time = time.time() - start

print(f"DROP partition:  {drop_time:.3f}s")
print(f"\nDrop is ~{delete_time/drop_time:.0f}x faster!")
print("Dropping a partition is a metadata operation — no row-by-row scanning.")
print("This is why time-series databases almost always use partitioning.")

# Re-create the January partition and restore data
cur.execute("""
    CREATE TABLE readings_2025_01 PARTITION OF sensor_readings
    FOR VALUES FROM ('2025-01-01') TO ('2025-02-01')
""")
cur.execute("""
    INSERT INTO sensor_readings (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM jan_backup
""")
cur.execute("DROP TABLE jan_backup")
print("\n(Re-created January 2025 partition with data restored.)")

```text
January 2025 readings: 37,200

DELETE 37,200 rows: 0.032s
DROP partition:  0.003s

Drop is ~10x faster!
Dropping a partition is a metadata operation — no row-by-row scanning.
This is why time-series databases almost always use partitioning.

(Re-created January 2025 partition.)
```

::: {.callout-important title="10x Faster Data Removal"}
| Method | Time |
|--------|---:|
| `DELETE ... WHERE` | 0.032s |
| `DROP PARTITION` | 0.003s |
:::

## Summary

| Strategy | Partition Key | Best For | Pruning Works With |
|----------|--------------|----------|-------------------|
| **Range** | `reading_time` | Time-series, date ranges | Range queries (`BETWEEN`, `>=`, `<`) |
| **List** | `unit` | Categories, regions, types | Equality queries (`=`, `IN`) |
| **Hash** | `sensor_id` | Even distribution, point lookups | Equality queries (`=`) only |

### Key Takeaways

1. **Partition pruning** is the main performance benefit --- queries only scan relevant partitions
2. **DROP partition is instant** vs row-by-row DELETE --- critical for data lifecycle management
3. **Choose your partition key** based on your most common query filter
4. All of this happens on a **single server** --- no network overhead, no distributed coordination

### Topics in the Comprehensive Lab

The full version also covers:

- Hash partitioning (even distribution, point-lookup pruning)
- PK constraint requirements (must include partition key)
- Missing partitions + DEFAULT partition safety net
- Per-partition indexes (smaller, cache-friendly)
- Per-partition maintenance (VACUUM/REINDEX)

In [ ]:
# Close the database connection
cur.close()
conn.close()
print("Connection closed.")
print("\nTo shut down Docker: cd partitioning-lab && docker compose down -v")

```text
Connection closed.

To shut down Docker: cd partitioning-lab && docker compose down -v
```